# Ngày 14 · Dự án P2: Titanic

- Dựng Pipeline xử lý thiếu, one-hot, chuẩn hóa và mô hình trong một đối tượng
- Đánh giá bằng stratified 5-fold có seed, không rò rỉ
- Xuất submission đúng định dạng Kaggle

**Đọc:**
- Pedregosa et al. (2011), Scikit-learn: Machine Learning in Python, JMLR 12
- ISLR, §5.1 (ôn) và §4.5

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.validation import check_is_fitted
NUMERIC, CATEGORICAL = ['Age', 'Fare', 'SibSp', 'Parch'], ['Sex', 'Pclass', 'Embarked']
def _titanic_like(n, seed):
    """Bảng giả lập cùng cấu trúc cột với Titanic (không phải dữ liệu thật) để kiểm tra pipeline."""
    r = np.random.default_rng(seed)
    df = pd.DataFrame({'Age': r.normal(30, 12, n).round(), 'Fare': r.gamma(2, 15, n).round(2), 'SibSp': r.integers(0, 4, n), 'Parch': r.integers(0, 3, n),
                       'Sex': r.choice(['male', 'female'], n), 'Pclass': r.choice([1, 2, 3], n), 'Embarked': r.choice(['S', 'C', 'Q'], n)})
    logit = -1 + 2.2 * (df.Sex == 'female') - .7 * (df.Pclass - 2) - .02 * (df.Age - 30)
    y = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df.loc[r.random(n) < .2, 'Age'] = np.nan
    df.loc[r.random(n) < .05, 'Embarked'] = np.nan
    return df, y
_df14, _y14 = _titanic_like(300, 14)
_test14 = _df14.head(5).copy(); _test14.loc[_test14.index[0], 'Embarked'] = 'X'; _test14.loc[_test14.index[1], 'Age'] = np.nan
def _unfitted(pipe):
    try:
        check_is_fitted(pipe)
        return False
    except NotFittedError:
        return True
def _ref_cv(make, C, seed):
    return cross_val_score(make(NUMERIC, CATEGORICAL, C), _df14, _y14, cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean()
def export_titanic(make, C=1.0):
    folder = DATA / 'kaggle/titanic'
    train, test = pd.read_csv(folder / 'train.csv'), pd.read_csv(folder / 'test.csv')
    pipe = make(NUMERIC, CATEGORICAL, C).fit(train[NUMERIC + CATEGORICAL], train.Survived)
    out = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': pipe.predict(test[NUMERIC + CATEGORICAL])})
    out.to_csv('titanic_submission.csv', index=False)
    return out


## Pipeline Titanic và cross-validation

Gợi ý: SimpleImputer(strategy="median") + StandardScaler cho cột số; SimpleImputer(strategy="most_frequent") + OneHotEncoder(handle_unknown="ignore") cho cột phân loại; LogisticRegression(C=C, max_iter=1000). cv_accuracy dùng StratifiedKFold(5, shuffle=True, random_state=seed).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def make_titanic_pipeline(numeric, categorical, C=1.0):
    # Trả Pipeline CHƯA fit.
    raise NotImplementedError

def cv_accuracy(make, X, y, C, seed):
    # Accuracy trung bình của stratified 5-fold (shuffle=True, random_state=seed).
    raise NotImplementedError


In [ ]:
_report('d14-pipeline',
    _check('Trả pipeline chưa fit', lambda: (_unfitted(make_titanic_pipeline(NUMERIC, CATEGORICAL)), 'Fit để cross_val_score làm trong từng fold.')),
    _check('Chịu được giá trị thiếu và nhãn mới ở test', lambda: (len(make_titanic_pipeline(NUMERIC, CATEGORICAL).fit(_df14, _y14).predict(_test14)) == 5, 'Embarked = "X" chưa từng gặp, Age bị thiếu.')),
    _check('cv_accuracy khớp stratified 5-fold tham chiếu', lambda: (np.isclose(cv_accuracy(make_titanic_pipeline, _df14, _y14, .5, 3), _ref_cv(make_titanic_pipeline, .5, 3)), 'C = 0,5, seed = 3.')),
)


## Dự án

Tải Titanic, dùng make_titanic_pipeline với cột số [Age, Fare, SibSp, Parch] và cột phân loại [Sex, Pclass, Embarked]. Chọn C bằng cv_accuracy, rồi chạy export_titanic(make_titanic_pipeline, C) để ghi titanic_submission.csv và tự nộp trên Kaggle. Báo cáo: bảng C → accuracy CV (trung bình ± độ lệch), điểm Kaggle, ba ca dự đoán sai và giả thuyết vì sao.